<a href="https://colab.research.google.com/github/lidwaa/cinetrack/blob/main/Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
benchmark/
├── config.py                    ← ⚠️ À configurer en premier
├── extractor.py                 ← Interface commune + tes wrappers
├── phase1_extract.py            ← Extraction sur les 50 docs
├── phase2a_cross_validation.py  ← Comparaison croisée entre modèles
├── phase2b_llm_judge.py         ← LLM-as-a-judge (Azure/OpenAI)
├── phase2c_manual_validation.py ← Validation manuelle interactive
├── phase3_gt_eval.py            ← Évaluation avec GT (CER + accuracy)
├── phase4_report.py             ← Rapport HTML + CSV final
└── run_all.py                   ← Lance tout d'un coup

In [ ]:
#---------extractor

"""
Interface d'extraction unifiée.

Flux par modèle :
  1. Prépare l'input selon ce que le modèle attend (PDF page 1 ou PNG)
  2. Lance l'OCR → texte brut
  3. Parse le texte via Mistral local → dict structuré JSON
  4. Écrit le résultat dans le CSV global
  5. Retourne (fields, latency_s, error)

Placeholders à remplacer :
  - pdf_to_png_first_page()  → ta fonction de conversion PDF→PNG
  - call_mistral_llm()       → ta logique d'appel Mistral medium 3
"""

import csv
import json
import tempfile
import time
import traceback
from pathlib import Path
from typing import Optional

import fitz  # pymupdf — pip install pymupdf

from config import ALL_VARS, RESULTS_DIR


# ─── CSV OUTPUT ───────────────────────────────────────────────────────────────

CSV_PATH   = RESULTS_DIR / "extractions.csv"
CSV_FIELDS = ["doc_id", "model", "latency_s", "ocr_latency_s",
              "llm_latency_s", "n_fields_extracted", "error"] + ALL_VARS


def _init_csv():
    """Crée le CSV avec header si inexistant."""
    CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    if not CSV_PATH.exists():
        with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=CSV_FIELDS).writeheader()


def _append_csv(doc_id: str, model: str, latency: float,
                ocr_lat: float, llm_lat: float,
                fields: dict, error: Optional[str]):
    row = {
        "doc_id":             doc_id,
        "model":              model,
        "latency_s":          round(latency, 3),
        "ocr_latency_s":      round(ocr_lat, 3),
        "llm_latency_s":      round(llm_lat, 3),
        "n_fields_extracted": len(fields),
        "error":              error or "",
    }
    for field in ALL_VARS:
        row[field] = fields.get(field, "")
    with open(CSV_PATH, "a", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=CSV_FIELDS).writerow(row)


# ─── PLACEHOLDERS ─────────────────────────────────────────────────────────────

def pdf_to_png_first_page(pdf_path: str) -> str:
    """
    ⚠️  PLACEHOLDER — Remplace par ta vraie fonction.

    Doit :
      - Prendre un chemin PDF en entrée
      - Convertir la PREMIÈRE PAGE en PNG
      - Retourner le chemin du PNG créé (fichier temporaire)

    Pour brancher ta fonction :
        from ton_module import ta_fonction
        def pdf_to_png_first_page(pdf_path: str) -> str:
            return ta_fonction(pdf_path)

    Implémentation de secours (PyMuPDF) active par défaut :
    """
    doc = fitz.open(pdf_path)
    page = doc[0]
    pix  = page.get_pixmap(matrix=fitz.Matrix(2, 2))  # x2 = meilleure qualité OCR
    tmp  = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    pix.save(tmp.name)
    doc.close()
    return tmp.name


def pdf_first_page_only(pdf_path: str) -> str:
    """
    Extrait la première page d'un PDF dans un fichier PDF temporaire.
    (utilisé par les modèles qui acceptent du PDF)
    """
    src      = fitz.open(pdf_path)
    page_doc = fitz.open()
    page_doc.insert_pdf(src, from_page=0, to_page=0)
    tmp = tempfile.NamedTemporaryFile(suffix=".pdf", delete=False)
    page_doc.save(tmp.name)
    src.close()
    page_doc.close()
    return tmp.name


def call_mistral_llm(ocr_text: str) -> dict:
    """
    ⚠️  PLACEHOLDER — Remplace par ta vraie logique d'appel Mistral medium 3.

    Doit :
      - Prendre le texte OCR brut en entrée
      - Appeler ton Mistral local avec le prompt ci-dessous (ou le tien)
      - Retourner un dict Python (pas une string JSON)

    Pour brancher ta logique :
        from ton_module import ton_appel_mistral
        def call_mistral_llm(ocr_text: str) -> dict:
            prompt = build_mistral_prompt(ocr_text)
            return ton_appel_mistral(prompt)  # doit retourner un dict

    Le LLM doit retourner un JSON de la forme :
        {
          "matricula":  "1234ABC",
          "titular":    "GARCIA LOPEZ JUAN",
          "nif":        "12345678Z",
          ...           (champs de ALL_VARS)
        }
    Les champs non trouvés doivent être null ou absents.
    """
    raise NotImplementedError(
        "⚠️  Remplace call_mistral_llm() par ta vraie logique d'appel Mistral.\n"
        "    Elle doit retourner un dict Python avec les champs extraits."
    )


def build_mistral_prompt(ocr_text: str) -> str:
    """Prompt à passer à Mistral pour l'extraction structurée."""
    fields_list = "\n".join(f"  - {f}" for f in ALL_VARS)
    return f"""Tu es un expert en lecture de cartes grises espagnoles (permis de circulation).

Voici le texte extrait par OCR d'une carte grise espagnole :

---
{ocr_text}
---

Extrait les informations suivantes si elles sont présentes dans le texte :
{fields_list}

Règles strictes :
- Réponds UNIQUEMENT en JSON valide, sans texte avant ou après, sans balises markdown.
- Si un champ est absent ou illisible, mets null.
- Normalise les valeurs : majuscules pour matricula, NIF, bastidor.
- Ne devine pas : si tu n'es pas sûr, mets null.

Format JSON attendu :
{json.dumps({f: None for f in ALL_VARS}, ensure_ascii=False, indent=2)}
"""


# ─── OCR PAR MODÈLE ───────────────────────────────────────────────────────────
# Chaque fonction prend le chemin de l'input préparé
# et retourne (texte_brut, latence_ocr_s, erreur_ou_None)

def _ocr_markitdown(input_path: str) -> tuple[str, float, Optional[str]]:
    """input_type : PDF (première page)"""
    try:
        from markitdown import MarkItDown
        t0     = time.perf_counter()
        result = MarkItDown().convert(input_path)
        return result.text_content, time.perf_counter() - t0, None
    except Exception:
        return "", 0.0, traceback.format_exc()


def _ocr_liteparse(input_path: str) -> tuple[str, float, Optional[str]]:
    """input_type : PDF (première page)"""
    try:
        import os
        os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
        from liteparse import LiteParse
        t0     = time.perf_counter()
        result = LiteParse().parse(input_path, ocr_enabled=True,
                                   ocr_language="es", max_pages=1)
        return result.text, time.perf_counter() - t0, None
    except Exception:
        return "", 0.0, traceback.format_exc()


def _ocr_paddleocr(input_path: str) -> tuple[str, float, Optional[str]]:
    """input_type : PNG"""
    try:
        import os
        os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
        from paddleocr import PaddleOCR

        # ⚠️  Adapte ce chemin
        MODELS_DIR = "/chemin/vers/paddleocr_models"

        t0     = time.perf_counter()
        ocr    = PaddleOCR(
            use_angle_cls=True, show_log=False,
            det_model_dir=f"{MODELS_DIR}/ch_PP-OCRv4_det_infer",
            rec_model_dir=f"{MODELS_DIR}/latin_PP-OCRv3_rec_infer",
            cls_model_dir=f"{MODELS_DIR}/ch_ppocr_mobile_v2.0_cls_infer",
        )
        result = ocr.ocr(input_path, cls=True)
        lines  = []
        if result and result[0]:
            for block in sorted(result[0], key=lambda b: b[0][0][1]):
                lines.append(block[1][0])
        return "\n".join(lines), time.perf_counter() - t0, None
    except Exception:
        return "", 0.0, traceback.format_exc()


def _ocr_docling(input_path: str) -> tuple[str, float, Optional[str]]:
    """input_type : PDF (première page)"""
    try:
        from docling.document_converter import DocumentConverter
        t0     = time.perf_counter()
        result = DocumentConverter().convert(input_path)
        return result.document.export_to_markdown(), time.perf_counter() - t0, None
    except Exception:
        return "", 0.0, traceback.format_exc()


def _ocr_glm(input_path: str) -> tuple[str, float, Optional[str]]:
    """
    ⚠️  PLACEHOLDER — Remplace par ta vraie intégration GLM OCR.
    input_type : PNG (à adapter si besoin)
    """
    try:
        raise NotImplementedError("GLM OCR pas encore intégré")
    except Exception:
        return "", 0.0, traceback.format_exc()


# ─── CONFIG MODÈLES ───────────────────────────────────────────────────────────
# input_type "pdf" → reçoit un PDF page 1 seule
# input_type "png" → reçoit un PNG de la page 1

MODEL_CONFIG = {
    "markitdown": {"input_type": "pdf", "ocr_fn": _ocr_markitdown},
    "liteparse":  {"input_type": "pdf", "ocr_fn": _ocr_liteparse},
    "paddleocr":  {"input_type": "png", "ocr_fn": _ocr_paddleocr},
    "docling":    {"input_type": "pdf", "ocr_fn": _ocr_docling},
    "glm_ocr":    {"input_type": "png", "ocr_fn": _ocr_glm},
}


# ─── POINT D'ENTRÉE PRINCIPAL ─────────────────────────────────────────────────

def extract(model_name: str, pdf_path: str) -> tuple[dict, float, Optional[str]]:
    """
    Point d'entrée unique pour tous les modèles.
    Retourne (fields_dict, total_latency_s, error_or_None).
    Écrit automatiquement une ligne dans results/extractions.csv.
    """
    _init_csv()
    doc_id    = Path(pdf_path).stem
    cfg       = MODEL_CONFIG.get(model_name)
    tmp_input = None

    if cfg is None:
        err = f"Modèle inconnu : {model_name}"
        _append_csv(doc_id, model_name, 0, 0, 0, {}, err)
        return {}, 0.0, err

    t_total = time.perf_counter()
    ocr_lat = llm_lat = 0.0

    try:
        # ── 1. Prépare l'input ─────────────────────────────────────────────
        if cfg["input_type"] == "pdf":
            tmp_input = pdf_first_page_only(pdf_path)
        else:
            tmp_input = pdf_to_png_first_page(pdf_path)

        # ── 2. OCR → texte brut ────────────────────────────────────────────
        ocr_text, ocr_lat, ocr_err = cfg["ocr_fn"](tmp_input)

        if ocr_err:
            _append_csv(doc_id, model_name,
                        time.perf_counter() - t_total, ocr_lat, 0, {}, ocr_err)
            return {}, ocr_lat, ocr_err

        if not ocr_text.strip():
            err = "OCR a retourné un texte vide"
            _append_csv(doc_id, model_name,
                        time.perf_counter() - t_total, ocr_lat, 0, {}, err)
            return {}, ocr_lat, err

        # ── 3. Parsing LLM ─────────────────────────────────────────────────
        t_llm = time.perf_counter()
        raw   = call_mistral_llm(ocr_text)
        llm_lat = time.perf_counter() - t_llm

        # Sécurité : si le LLM retourne une string au lieu d'un dict
        if isinstance(raw, str):
            raw = raw.strip().lstrip("```json").rstrip("```").strip()
            raw = json.loads(raw)

        # ── 4. Filtre les champs valides ───────────────────────────────────
        fields = {
            k: str(v).strip()
            for k, v in raw.items()
            if k in ALL_VARS and v is not None and str(v).strip() not in ("", "null")
        }

        total_lat = time.perf_counter() - t_total
        _append_csv(doc_id, model_name, total_lat, ocr_lat, llm_lat, fields, None)
        return fields, total_lat, None

    except NotImplementedError as e:
        err = str(e)
        _append_csv(doc_id, model_name,
                    time.perf_counter() - t_total, ocr_lat, llm_lat, {}, err)
        return {}, 0.0, err

    except json.JSONDecodeError as e:
        err = f"Erreur parsing JSON LLM : {e}"
        _append_csv(doc_id, model_name,
                    time.perf_counter() - t_total, ocr_lat, llm_lat, {}, err)
        return {}, time.perf_counter() - t_total, err

    except Exception:
        err = traceback.format_exc()
        _append_csv(doc_id, model_name,
                    time.perf_counter() - t_total, ocr_lat, llm_lat, {}, err)
        return {}, time.perf_counter() - t_total, err

    finally:
        if tmp_input:
            Path(tmp_input).unlink(missing_ok=True)

In [ ]:
#-----config

"""
Configuration centrale du benchmark OCR - Cartes grises espagnoles
"""

from pathlib import Path

# ─── Répertoires ──────────────────────────────────────────────────────────────
BASE_DIR         = Path(__file__).parent
PDFS_DIR         = BASE_DIR / "pdfs"
GROUND_TRUTH_DIR = BASE_DIR / "ground_truth"
RESULTS_DIR      = BASE_DIR / "results"
RESULTS_RAW_DIR  = RESULTS_DIR / "raw"
RESULTS_COMP_DIR = RESULTS_DIR / "comparison"
RESULTS_GT_DIR   = RESULTS_DIR / "gt_eval"
RESULTS_REPORT   = RESULTS_DIR / "report"

for d in [RESULTS_RAW_DIR, RESULTS_COMP_DIR, RESULTS_GT_DIR, RESULTS_REPORT]:
    d.mkdir(parents=True, exist_ok=True)

# ─── Modèles actifs ───────────────────────────────────────────────────────────
# Commente les modèles que tu ne veux pas exécuter
ACTIVE_MODELS = [
    "markitdown",
    "liteparse",
    "paddleocr",
    "docling",
    # "glm_ocr",  # décommente quand prêt
]

# ─── Champs à extraire ────────────────────────────────────────────────────────
# Remplace par ton ALL_VARS réel
ALL_VARS = [
    "matricula",
    "titular",
    "nif",
    "marca",
    "modelo",
    "bastidor",
    "fecha_matriculacion",
    "direccion",
    # Ajoute ici tous tes champs
]

# ─── LLM-as-a-judge (Azure OpenAI) ───────────────────────────────────────────
LLM_JUDGE_CONFIG = {
    "api_type":    "azure",                          # ou "openai"
    "api_base":    "https://TON-ENDPOINT.openai.azure.com/",
    "api_key":     "TA_CLE_API",
    "api_version": "2024-02-01",
    "deployment":  "gpt-4o",                        # nom de ton déploiement
}

# ─── Validation croisée ───────────────────────────────────────────────────────
# Nombre minimum de modèles en accord pour considérer un champ comme "consensus"
CONSENSUS_MIN = 2  # sur N modèles actifs

# ─── Timeouts ─────────────────────────────────────────────────────────────────
EXTRACTION_TIMEOUT_S = 120  # secondes max par doc par modèle

In [ ]:
"""
Phase 1 — Extraction brute sur tous les PDFs.
Lance chaque modèle sur chaque doc et sauvegarde les résultats raw.

Usage :
    python phase1_extract.py
    python phase1_extract.py --models markitdown liteparse   # subset
    python phase1_extract.py --resume                        # skip docs déjà traités
"""

import argparse
import json
import sys
from pathlib import Path
from datetime import datetime

from config import PDFS_DIR, RESULTS_RAW_DIR, ACTIVE_MODELS
from extractor import extract


def run(models: list[str], resume: bool):
    pdfs = sorted(PDFS_DIR.glob("*.pdf"))
    if not pdfs:
        print(f"❌ Aucun PDF trouvé dans {PDFS_DIR}")
        sys.exit(1)

    print(f"📂 {len(pdfs)} PDFs  |  Modèles : {models}\n")

    # Stats globales
    stats = {m: {"ok": 0, "err": 0, "total_s": 0.0} for m in models}

    for pdf in pdfs:
        doc_id = pdf.stem
        print(f"─── {doc_id} ───────────────────────────")

        for model in models:
            out_path = RESULTS_RAW_DIR / model / f"{doc_id}.json"
            out_path.parent.mkdir(parents=True, exist_ok=True)

            # Resume : skip si déjà traité
            if resume and out_path.exists():
                print(f"  [{model}] ⏭  déjà traité")
                continue

            fields, latency, error = extract(model, str(pdf))

            result = {
                "doc_id":    doc_id,
                "model":     model,
                "timestamp": datetime.now().isoformat(),
                "latency_s": round(latency, 3),
                "fields":    fields,
                "error":     error,
                "n_fields_extracted": len(fields),
            }

            with open(out_path, "w", encoding="utf-8") as f:
                json.dump(result, f, ensure_ascii=False, indent=2)

            if error:
                stats[model]["err"] += 1
                print(f"  [{model}] ❌ {error.splitlines()[-1]}")
            else:
                stats[model]["ok"] += 1
                stats[model]["total_s"] += latency
                print(f"  [{model}] ✅ {len(fields)} champs  |  {latency:.2f}s")

    # ─── Résumé ───────────────────────────────────────────────────────────────
    print("\n" + "═" * 50)
    print(f"{'MODÈLE':<15} {'OK':>5} {'ERR':>5} {'MOY (s)':>10}")
    print("─" * 50)
    for model in models:
        s = stats[model]
        avg = s["total_s"] / s["ok"] if s["ok"] else 0
        print(f"{model:<15} {s['ok']:>5} {s['err']:>5} {avg:>10.2f}s")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--models", nargs="+", default=ACTIVE_MODELS)
    parser.add_argument("--resume", action="store_true",
                        help="Skip les docs déjà traités")
    args = parser.parse_args()
    run(args.models, args.resume)

In [ ]:
"""
Phase 2a — Validation croisée entre modèles.
Pour chaque doc et chaque champ, compare les valeurs extraites par tous les modèles.
Détecte les consensus et les divergences.

Usage :
    python phase2a_cross_validation.py
"""

import json
from collections import Counter
from pathlib import Path

from config import RESULTS_RAW_DIR, RESULTS_COMP_DIR, ACTIVE_MODELS, ALL_VARS, CONSENSUS_MIN


def normalize(val: str) -> str:
    """Normalise une valeur pour comparaison (casse, espaces)."""
    return val.upper().strip() if val else ""


def load_raw(model: str, doc_id: str) -> dict:
    p = RESULTS_RAW_DIR / model / f"{doc_id}.json"
    if not p.exists():
        return {}
    with open(p) as f:
        data = json.load(f)
    return data.get("fields", {})


def cross_validate_doc(doc_id: str, models: list[str]) -> dict:
    """Analyse croisée pour un doc — retourne le rapport."""
    # Charge les extractions de chaque modèle
    extractions = {m: load_raw(m, doc_id) for m in models}

    field_analysis = {}
    for field in ALL_VARS:
        values = {}
        for model in models:
            val = extractions[model].get(field)
            if val:
                values[model] = normalize(val)

        if not values:
            field_analysis[field] = {
                "status":    "missing_all",
                "consensus": None,
                "values":    {},
                "agreement": 0,
            }
            continue

        # Compte les valeurs identiques
        counter = Counter(values.values())
        most_common_val, most_common_count = counter.most_common(1)[0]

        # Modèles qui ont cette valeur
        agreeing_models = [m for m, v in values.items() if v == most_common_val]

        status = "consensus" if most_common_count >= CONSENSUS_MIN else "divergence"
        if len(values) == 1:
            status = "single"  # un seul modèle a extrait ce champ

        field_analysis[field] = {
            "status":          status,
            "consensus":       most_common_val if status == "consensus" else None,
            "values":          values,          # {model: valeur}
            "agreement":       most_common_count,
            "agreeing_models": agreeing_models,
        }

    # Score global du doc
    n_consensus  = sum(1 for f in field_analysis.values() if f["status"] == "consensus")
    n_divergence = sum(1 for f in field_analysis.values() if f["status"] == "divergence")
    n_missing    = sum(1 for f in field_analysis.values() if f["status"] == "missing_all")
    n_single     = sum(1 for f in field_analysis.values() if f["status"] == "single")

    return {
        "doc_id":       doc_id,
        "models":       models,
        "fields":       field_analysis,
        "summary": {
            "consensus":  n_consensus,
            "divergence": n_divergence,
            "single":     n_single,
            "missing":    n_missing,
            "total":      len(ALL_VARS),
        },
    }


def run():
    # Récupère tous les doc_ids disponibles
    doc_ids = set()
    for model in ACTIVE_MODELS:
        model_dir = RESULTS_RAW_DIR / model
        if model_dir.exists():
            doc_ids.update(p.stem for p in model_dir.glob("*.json"))

    if not doc_ids:
        print("❌ Aucun résultat raw trouvé. Lance d'abord phase1_extract.py")
        return

    print(f"📊 Validation croisée sur {len(doc_ids)} docs  |  Modèles : {ACTIVE_MODELS}\n")

    all_summaries = []

    for doc_id in sorted(doc_ids):
        report = cross_validate_doc(doc_id, ACTIVE_MODELS)
        out_path = RESULTS_COMP_DIR / f"{doc_id}_cross.json"
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(report, f, ensure_ascii=False, indent=2)

        s = report["summary"]
        print(f"  {doc_id:<30} ✅ {s['consensus']} consensus  "
              f"⚠️  {s['divergence']} divergences  "
              f"❓ {s['single']} single  "
              f"❌ {s['missing']} manquants")
        all_summaries.append(report)

    # ─── Résumé global par modèle ─────────────────────────────────────────────
    print("\n" + "═" * 60)
    print("TAUX DE REMPLISSAGE PAR MODÈLE (champs non vides / total attendu)")
    print("─" * 60)

    for model in ACTIVE_MODELS:
        filled = 0
        total  = 0
        for rep in all_summaries:
            for field, fa in rep["fields"].items():
                total += 1
                if model in fa.get("values", {}):
                    filled += 1
        rate = filled / total * 100 if total else 0
        print(f"  {model:<15} {filled:>5}/{total:<5}  ({rate:.1f}%)")

    # Sauvegarde résumé global
    summary_path = RESULTS_COMP_DIR / "cross_summary.json"
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(all_summaries, f, ensure_ascii=False, indent=2)
    print(f"\n💾 Résumé sauvegardé → {summary_path}")


if __name__ == "__main__":
    run()

In [ ]:
"""
Phase 2b — LLM-as-a-judge via Azure OpenAI.
Pour chaque doc avec divergences, envoie les extractions à un LLM
qui évalue laquelle est la plus correcte champ par champ.

Usage :
    python phase2b_llm_judge.py
    python phase2b_llm_judge.py --only-divergences   # skip les consensus (plus rapide)
    python phase2b_llm_judge.py --max-docs 10        # test sur 10 docs
"""

import argparse
import json
import time
from pathlib import Path

from config import (
    PDFS_DIR, RESULTS_RAW_DIR, RESULTS_COMP_DIR,
    ACTIVE_MODELS, ALL_VARS, LLM_JUDGE_CONFIG
)


# ─── Client Azure OpenAI ──────────────────────────────────────────────────────

def get_llm_client():
    """Initialise le client Azure OpenAI ou OpenAI standard."""
    try:
        from openai import AzureOpenAI, OpenAI
        cfg = LLM_JUDGE_CONFIG
        if cfg["api_type"] == "azure":
            return AzureOpenAI(
                azure_endpoint=cfg["api_base"],
                api_key=cfg["api_key"],
                api_version=cfg["api_version"],
            ), cfg["deployment"]
        else:
            return OpenAI(api_key=cfg["api_key"]), cfg["deployment"]
    except ImportError:
        raise ImportError("pip install openai")


# ─── Prompt ───────────────────────────────────────────────────────────────────

JUDGE_SYSTEM = """Tu es un expert en lecture de cartes grises espagnoles (permis de circulation).
Tu reçois les extractions de plusieurs modèles OCR pour un même document.
Ta mission : pour chaque champ, identifier quelle extraction est la plus correcte,
ou proposer une valeur corrigée si toutes les extractions semblent partiellement incorrectes.
Réponds UNIQUEMENT en JSON valide, sans texte avant ou après."""

JUDGE_PROMPT = """Voici les extractions OCR de {n_models} modèles pour le document "{doc_id}".

EXTRACTIONS PAR MODÈLE:
{extractions_text}

CHAMPS ATTENDUS: {fields}

Pour chaque champ, réponds avec:
- "best_model": le modèle dont la valeur est la plus correcte (ou null si aucune)
- "best_value": la valeur correcte (tu peux corriger si nécessaire)
- "confidence": "high" / "medium" / "low"
- "note": explication courte si pertinent

Réponds en JSON avec cette structure exacte:
{{
  "champ1": {{"best_model": "...", "best_value": "...", "confidence": "...", "note": "..."}},
  "champ2": ...
}}
"""


def build_extractions_text(extractions: dict) -> str:
    lines = []
    for model, fields in extractions.items():
        lines.append(f"\n[{model.upper()}]")
        if not fields:
            lines.append("  (aucune extraction)")
        for field, val in fields.items():
            lines.append(f"  {field}: {val}")
    return "\n".join(lines)


# ─── Judge ────────────────────────────────────────────────────────────────────

def judge_doc(client, deployment: str, doc_id: str,
              extractions: dict, only_divergences: bool) -> dict:
    """Appelle le LLM pour juger les extractions d'un doc."""

    # Si only_divergences, charge le rapport cross pour filtrer les champs
    divergent_fields = ALL_VARS
    if only_divergences:
        cross_path = RESULTS_COMP_DIR / f"{doc_id}_cross.json"
        if cross_path.exists():
            with open(cross_path) as f:
                cross = json.load(f)
            divergent_fields = [
                field for field, fa in cross["fields"].items()
                if fa["status"] in ("divergence", "single", "missing_all")
            ]

    if not divergent_fields:
        return {"doc_id": doc_id, "status": "skipped_all_consensus", "judgments": {}}

    # Filtre les extractions aux champs divergents seulement
    filtered_extractions = {
        model: {k: v for k, v in fields.items() if k in divergent_fields}
        for model, fields in extractions.items()
    }

    prompt = JUDGE_PROMPT.format(
        n_models=len(extractions),
        doc_id=doc_id,
        extractions_text=build_extractions_text(filtered_extractions),
        fields=", ".join(divergent_fields),
    )

    try:
        response = client.chat.completions.create(
            model=deployment,
            messages=[
                {"role": "system", "content": JUDGE_SYSTEM},
                {"role": "user",   "content": prompt},
            ],
            temperature=0,
            max_tokens=2000,
        )
        raw = response.choices[0].message.content.strip()
        # Nettoie les éventuels backticks
        raw = raw.replace("```json", "").replace("```", "").strip()
        judgments = json.loads(raw)
        return {"doc_id": doc_id, "status": "ok", "judgments": judgments,
                "judged_fields": divergent_fields}
    except json.JSONDecodeError as e:
        return {"doc_id": doc_id, "status": "parse_error", "error": str(e),
                "raw_response": raw}
    except Exception as e:
        return {"doc_id": doc_id, "status": "api_error", "error": str(e)}


# ─── Main ─────────────────────────────────────────────────────────────────────

def run(only_divergences: bool, max_docs: int):
    client, deployment = get_llm_client()

    # Récupère les doc_ids
    doc_ids = set()
    for model in ACTIVE_MODELS:
        model_dir = RESULTS_RAW_DIR / model
        if model_dir.exists():
            doc_ids.update(p.stem for p in model_dir.glob("*.json"))

    doc_ids = sorted(doc_ids)[:max_docs] if max_docs else sorted(doc_ids)
    print(f"🤖 LLM-as-a-judge sur {len(doc_ids)} docs\n")

    results = []
    for i, doc_id in enumerate(doc_ids, 1):
        # Charge toutes les extractions
        extractions = {}
        for model in ACTIVE_MODELS:
            raw_path = RESULTS_RAW_DIR / model / f"{doc_id}.json"
            if raw_path.exists():
                with open(raw_path) as f:
                    data = json.load(f)
                extractions[model] = data.get("fields", {})

        print(f"  [{i}/{len(doc_ids)}] {doc_id} ...", end=" ", flush=True)
        t0 = time.perf_counter()
        result = judge_doc(client, deployment, doc_id, extractions, only_divergences)
        elapsed = time.perf_counter() - t0

        status_icon = "✅" if result["status"] == "ok" else "⚠️ " if result["status"] == "skipped_all_consensus" else "❌"
        n_judged = len(result.get("judged_fields", []))
        print(f"{status_icon} {n_judged} champs jugés  |  {elapsed:.1f}s")

        # Sauvegarde
        out_path = RESULTS_COMP_DIR / f"{doc_id}_llm_judge.json"
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        results.append(result)
        time.sleep(0.5)  # Évite le rate limiting

    # Résumé par modèle : combien de fois chaque modèle a été jugé "best"
    model_wins = {m: 0 for m in ACTIVE_MODELS}
    total_judgments = 0
    for r in results:
        for field, judgment in r.get("judgments", {}).items():
            best = judgment.get("best_model")
            if best and best in model_wins:
                model_wins[best] += 1
                total_judgments += 1

    print("\n" + "═" * 50)
    print("SCORE LLM-JUDGE (nombre de fois 'best model')")
    print("─" * 50)
    for model, wins in sorted(model_wins.items(), key=lambda x: -x[1]):
        pct = wins / total_judgments * 100 if total_judgments else 0
        print(f"  {model:<15} {wins:>5} fois  ({pct:.1f}%)")

    # Sauvegarde résumé
    summary_path = RESULTS_COMP_DIR / "llm_judge_summary.json"
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump({"model_wins": model_wins, "details": results}, f,
                  ensure_ascii=False, indent=2)
    print(f"\n💾 Résumé LLM-judge → {summary_path}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--only-divergences", action="store_true",
                        help="Ne juge que les champs en désaccord entre modèles")
    parser.add_argument("--max-docs", type=int, default=0,
                        help="Limite le nombre de docs (0 = tous)")
    args = parser.parse_args()
    run(args.only_divergences, args.max_docs)

In [ ]:
"""
Phase 2c — Validation manuelle interactive.
Affiche les extractions de chaque modèle et te permet de valider/corriger
champ par champ. Sauvegarde un ground truth partiel.

Usage :
    python phase2c_manual_validation.py
    python phase2c_manual_validation.py --doc carte_001   # un doc spécifique
    python phase2c_manual_validation.py --only-divergences
"""

import argparse
import json
import os
from pathlib import Path

from config import (
    RESULTS_RAW_DIR, RESULTS_COMP_DIR, GROUND_TRUTH_DIR,
    ACTIVE_MODELS, ALL_VARS
)

GROUND_TRUTH_DIR.mkdir(parents=True, exist_ok=True)


def color(text: str, code: str) -> str:
    """Colorisation terminal."""
    codes = {"green": "32", "yellow": "33", "red": "31", "cyan": "36", "bold": "1"}
    return f"\033[{codes.get(code, '0')}m{text}\033[0m"


def show_doc_extractions(doc_id: str) -> dict:
    """Affiche les extractions de tous les modèles pour un doc."""
    extractions = {}
    for model in ACTIVE_MODELS:
        raw_path = RESULTS_RAW_DIR / model / f"{doc_id}.json"
        if raw_path.exists():
            with open(raw_path) as f:
                data = json.load(f)
            extractions[model] = data.get("fields", {})
        else:
            extractions[model] = {}
    return extractions


def validate_doc(doc_id: str, only_divergences: bool) -> dict:
    """Validation manuelle interactive pour un doc."""
    print("\n" + "═" * 70)
    print(color(f"  DOC : {doc_id}", "bold"))
    print("═" * 70)

    extractions = show_doc_extractions(doc_id)

    # Charge le rapport cross si dispo
    cross_data = {}
    cross_path = RESULTS_COMP_DIR / f"{doc_id}_cross.json"
    if cross_path.exists():
        with open(cross_path) as f:
            cross_data = json.load(f).get("fields", {})

    gt = {}
    skipped = 0

    for field in ALL_VARS:
        fa = cross_data.get(field, {})
        status = fa.get("status", "unknown")

        # Skip consensus si only_divergences
        if only_divergences and status == "consensus":
            gt[field] = fa.get("consensus")
            skipped += 1
            continue

        # Affiche les valeurs par modèle
        print(f"\n  {color(field.upper(), 'cyan')}", end="")
        if status == "consensus":
            print(f"  {color('[CONSENSUS]', 'green')}", end="")
        elif status == "divergence":
            print(f"  {color('[DIVERGENCE]', 'yellow')}", end="")
        elif status == "missing_all":
            print(f"  {color('[MANQUANT]', 'red')}", end="")
        print()

        values_shown = {}
        for model in ACTIVE_MODELS:
            val = extractions[model].get(field, "")
            values_shown[model] = val
            icon = "✅" if val else "  "
            print(f"    {icon} {model:<15} : {color(val or '(vide)', 'yellow' if not val else 'green')}")

        # Détermine la valeur suggérée (consensus ou valeur la plus fréquente)
        suggestion = fa.get("consensus") or ""
        if not suggestion:
            non_empty = [v for v in values_shown.values() if v]
            if non_empty:
                suggestion = max(set(non_empty), key=non_empty.count)

        # Demande validation
        print(f"\n  Valeur correcte ? ", end="")
        if suggestion:
            print(f"[Entrée = '{color(suggestion, 'green')}', "
                  f"'-' = vide, ou tape la valeur] : ", end="")
        else:
            print(f"[Entrée = vide, ou tape la valeur] : ", end="")

        user_input = input().strip()

        if user_input == "-":
            gt[field] = None
        elif user_input == "" and suggestion:
            gt[field] = suggestion
        elif user_input == "":
            gt[field] = None
        else:
            gt[field] = user_input

    print(f"\n  ⏭  {skipped} champs en consensus auto-validés")
    return {k: v for k, v in gt.items() if v is not None}


def run(target_doc: str, only_divergences: bool):
    # Récupère les doc_ids
    doc_ids = set()
    for model in ACTIVE_MODELS:
        model_dir = RESULTS_RAW_DIR / model
        if model_dir.exists():
            doc_ids.update(p.stem for p in model_dir.glob("*.json"))

    if target_doc:
        doc_ids = {target_doc} if target_doc in doc_ids else set()
        if not doc_ids:
            print(f"❌ Doc '{target_doc}' introuvable dans les résultats")
            return

    doc_ids = sorted(doc_ids)

    # Skip docs déjà validés
    pending = [d for d in doc_ids
               if not (GROUND_TRUTH_DIR / f"{d}.json").exists()]

    print(f"📋 {len(pending)} docs à valider ({len(doc_ids) - len(pending)} déjà faits)")
    if not pending:
        print("✅ Tous les docs sont déjà validés !")
        return

    print("  [Ctrl+C pour arrêter et sauvegarder la progression]\n")

    validated = 0
    try:
        for doc_id in pending:
            gt = validate_doc(doc_id, only_divergences)

            # Sauvegarde ground truth
            gt_path = GROUND_TRUTH_DIR / f"{doc_id}.json"
            with open(gt_path, "w", encoding="utf-8") as f:
                json.dump(gt, f, ensure_ascii=False, indent=2)

            print(f"\n  💾 GT sauvegardé → {gt_path}  ({len(gt)} champs)\n")
            validated += 1

            # Continue ?
            if validated < len(pending):
                cont = input("  Prochain doc ? [Entrée=oui / 'q'=quitter] : ").strip()
                if cont.lower() == "q":
                    break

    except KeyboardInterrupt:
        print("\n\n  ⏹  Arrêt — progression sauvegardée")

    print(f"\n✅ {validated} docs validés sur {len(pending)}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--doc", type=str, default="",
                        help="Valider un doc spécifique")
    parser.add_argument("--only-divergences", action="store_true",
                        help="Auto-valide les consensus, ne demande que les divergences")
    args = parser.parse_args()
    run(args.doc, args.only_divergences)

In [ ]:
"""
Phase 3 — Évaluation avec ground truth.
Calcule CER + exact match pour les docs qui ont un GT (validé manuellement ou fourni).

Usage :
    python phase3_gt_eval.py
"""

import json
import csv
from pathlib import Path
from datetime import datetime

from config import RESULTS_RAW_DIR, RESULTS_GT_DIR, GROUND_TRUTH_DIR, ACTIVE_MODELS, ALL_VARS


# ─── Métriques ────────────────────────────────────────────────────────────────

def cer(ref: str, hyp: str) -> float:
    """Character Error Rate."""
    ref = (ref or "").upper().strip()
    hyp = (hyp or "").upper().strip()
    if not ref:
        return 0.0 if not hyp else 1.0
    m, n = len(ref), len(hyp)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, n + 1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev[j-1] + cost)
    return round(dp[n] / m, 4)


def run():
    gt_files = sorted(GROUND_TRUTH_DIR.glob("*.json"))
    if not gt_files:
        print("❌ Aucun ground truth trouvé dans", GROUND_TRUTH_DIR)
        print("   Lance d'abord phase2c_manual_validation.py")
        return

    print(f"📐 Évaluation GT sur {len(gt_files)} docs  |  Modèles : {ACTIVE_MODELS}\n")

    all_rows = []
    # Accumulateurs pour résumé global
    model_stats = {
        m: {f: {"exact": 0, "cer_sum": 0.0, "n": 0} for f in ALL_VARS}
        for m in ACTIVE_MODELS
    }

    for gt_file in gt_files:
        doc_id = gt_file.stem
        with open(gt_file) as f:
            gt = json.load(f)

        print(f"  {doc_id}")
        for model in ACTIVE_MODELS:
            raw_path = RESULTS_RAW_DIR / model / f"{doc_id}.json"
            if not raw_path.exists():
                continue
            with open(raw_path) as f:
                data = json.load(f)
            extracted = data.get("fields", {})
            latency   = data.get("latency_s", None)

            for field in ALL_VARS:
                ref = gt.get(field)
                if ref is None:
                    continue  # champ non annoté dans ce GT

                hyp         = extracted.get(field, "")
                exact       = (ref.upper().strip() == (hyp or "").upper().strip())
                cer_val     = cer(ref, hyp)

                model_stats[model][field]["exact"]   += int(exact)
                model_stats[model][field]["cer_sum"] += cer_val
                model_stats[model][field]["n"]       += 1

                all_rows.append({
                    "doc_id":      doc_id,
                    "model":       model,
                    "field":       field,
                    "ref":         ref,
                    "hyp":         hyp or "",
                    "exact_match": exact,
                    "cer":         cer_val,
                    "latency_s":   latency,
                })

    # ─── Export CSV détaillé ──────────────────────────────────────────────────
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = RESULTS_GT_DIR / f"gt_eval_detail_{ts}.csv"
    if all_rows:
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=all_rows[0].keys())
            writer.writeheader()
            writer.writerows(all_rows)
        print(f"\n💾 Détail → {csv_path}")

    # ─── Résumé console ───────────────────────────────────────────────────────
    print("\n" + "═" * 65)
    print(f"{'MODÈLE':<15} {'CHAMP':<25} {'ACCURACY':>10} {'CER MOY':>10} {'N':>5}")
    print("─" * 65)

    summary_rows = []
    for model in ACTIVE_MODELS:
        for field in ALL_VARS:
            s = model_stats[model][field]
            if s["n"] == 0:
                continue
            acc     = s["exact"] / s["n"] * 100
            avg_cer = s["cer_sum"] / s["n"]
            print(f"{model:<15} {field:<25} {acc:>9.1f}%  {avg_cer:>9.4f}  {s['n']:>5}")
            summary_rows.append({
                "model": model, "field": field,
                "accuracy_pct": round(acc, 2),
                "avg_cer": round(avg_cer, 4),
                "n": s["n"],
            })
        print("─" * 65)

    # Résumé global par modèle (toutes champs confondus)
    print("\nRÉSUMÉ GLOBAL PAR MODÈLE")
    print("─" * 40)
    for model in ACTIVE_MODELS:
        total_exact = sum(model_stats[model][f]["exact"] for f in ALL_VARS)
        total_n     = sum(model_stats[model][f]["n"]     for f in ALL_VARS)
        total_cer   = sum(model_stats[model][f]["cer_sum"] for f in ALL_VARS)
        if total_n == 0:
            continue
        print(f"  {model:<15}  Accuracy: {total_exact/total_n*100:>5.1f}%  "
              f"CER moy: {total_cer/total_n:.4f}  ({total_n} champs évalués)")

    # Sauvegarde résumé JSON
    summary_path = RESULTS_GT_DIR / f"gt_eval_summary_{ts}.json"
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary_rows, f, ensure_ascii=False, indent=2)
    print(f"\n📊 Résumé → {summary_path}")


if __name__ == "__main__":
    run()

In [ ]:
"""
Phase 4 — Rapport final.
Agrège tous les résultats (latence, cross-validation, LLM-judge, GT eval)
et génère un rapport HTML interactif + CSV récapitulatif.

Usage :
    python phase4_report.py
"""

import json
import csv
from pathlib import Path
from datetime import datetime
from collections import defaultdict

from config import (
    RESULTS_RAW_DIR, RESULTS_COMP_DIR, RESULTS_GT_DIR,
    RESULTS_REPORT, ACTIVE_MODELS, ALL_VARS
)


def load_latencies() -> dict:
    """Latence moyenne par modèle."""
    latencies = defaultdict(list)
    for model in ACTIVE_MODELS:
        model_dir = RESULTS_RAW_DIR / model
        if not model_dir.exists():
            continue
        for p in model_dir.glob("*.json"):
            with open(p) as f:
                data = json.load(f)
            if data.get("latency_s"):
                latencies[model].append(data["latency_s"])
    return {m: (sum(v)/len(v) if v else 0) for m, v in latencies.items()}


def load_fill_rates() -> dict:
    """Taux de remplissage par modèle (% de champs non vides)."""
    filled = defaultdict(int)
    total  = defaultdict(int)
    for model in ACTIVE_MODELS:
        model_dir = RESULTS_RAW_DIR / model
        if not model_dir.exists():
            continue
        for p in model_dir.glob("*.json"):
            with open(p) as f:
                data = json.load(f)
            fields = data.get("fields", {})
            for field in ALL_VARS:
                total[model] += 1
                if fields.get(field):
                    filled[model] += 1
    return {m: (filled[m]/total[m]*100 if total[m] else 0) for m in ACTIVE_MODELS}


def load_llm_scores() -> dict:
    """Score LLM-judge par modèle."""
    summary_path = RESULTS_COMP_DIR / "llm_judge_summary.json"
    if not summary_path.exists():
        return {}
    with open(summary_path) as f:
        data = json.load(f)
    wins = data.get("model_wins", {})
    total = sum(wins.values())
    return {m: (wins.get(m, 0)/total*100 if total else 0) for m in ACTIVE_MODELS}


def load_gt_scores() -> dict:
    """Accuracy GT par modèle."""
    gt_files = sorted(RESULTS_GT_DIR.glob("gt_eval_summary_*.json"))
    if not gt_files:
        return {}
    with open(gt_files[-1]) as f:  # Dernier run
        rows = json.load(f)
    model_acc = defaultdict(list)
    for row in rows:
        model_acc[row["model"]].append(row["accuracy_pct"])
    return {m: (sum(v)/len(v) if v else 0) for m, v in model_acc.items()}


def generate_html_report(data: dict) -> str:
    models      = data["models"]
    latencies   = data["latencies"]
    fill_rates  = data["fill_rates"]
    llm_scores  = data["llm_scores"]
    gt_scores   = data["gt_scores"]
    generated   = data["generated"]

    # Couleurs par modèle
    colors = ["#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f"]
    model_colors = {m: colors[i % len(colors)] for i, m in enumerate(models)}

    def bar(value: float, max_val: float, color: str, suffix: str = "%") -> str:
        pct = min(value / max_val * 100, 100) if max_val else 0
        return f'''<div class="bar-wrap">
            <div class="bar" style="width:{pct:.0f}%;background:{color}"></div>
            <span class="bar-val">{value:.1f}{suffix}</span>
        </div>'''

    max_lat  = max(latencies.values(), default=1)
    rows_lat = "".join(
        f"<tr><td>{m}</td><td>{bar(latencies.get(m,0), max_lat, model_colors[m], 's')}</td></tr>"
        for m in models
    )
    rows_fill = "".join(
        f"<tr><td>{m}</td><td>{bar(fill_rates.get(m,0), 100, model_colors[m])}</td></tr>"
        for m in models
    )
    rows_llm = "".join(
        f"<tr><td>{m}</td><td>{bar(llm_scores.get(m,0), 100, model_colors[m])}</td></tr>"
        for m in models
    ) if llm_scores else "<tr><td colspan='2'>Non disponible</td></tr>"

    rows_gt = "".join(
        f"<tr><td>{m}</td><td>{bar(gt_scores.get(m,0), 100, model_colors[m])}</td></tr>"
        for m in models
    ) if gt_scores else "<tr><td colspan='2'>Non disponible (pas de GT)</td></tr>"

    return f"""<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<title>Benchmark OCR — Cartes grises ES</title>
<style>
  body {{ font-family: 'Segoe UI', sans-serif; margin: 0; background: #f5f7fa; color: #333; }}
  header {{ background: #1a2744; color: white; padding: 24px 40px; }}
  header h1 {{ margin: 0; font-size: 1.6em; }}
  header p  {{ margin: 4px 0 0; opacity: .7; font-size:.9em; }}
  .container {{ max-width: 1100px; margin: 30px auto; padding: 0 20px; }}
  .grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 24px; }}
  .card {{ background: white; border-radius: 10px; padding: 24px;
           box-shadow: 0 2px 8px rgba(0,0,0,.08); }}
  .card h2 {{ margin: 0 0 18px; font-size: 1.1em; color: #1a2744;
              border-bottom: 2px solid #e8ecf0; padding-bottom: 10px; }}
  table {{ width: 100%; border-collapse: collapse; }}
  td {{ padding: 8px 6px; vertical-align: middle; font-size: .9em; }}
  td:first-child {{ width: 130px; font-weight: 600; color: #555; }}
  .bar-wrap {{ display: flex; align-items: center; gap: 10px; }}
  .bar {{ height: 22px; border-radius: 4px; min-width: 4px;
          transition: width .3s; }}
  .bar-val {{ font-size: .85em; color: #555; white-space: nowrap; }}
  .badge {{ display: inline-block; padding: 2px 10px; border-radius: 12px;
            font-size:.8em; font-weight:600; color:white; }}
  .models {{ display: flex; flex-wrap: wrap; gap: 10px; margin: 16px 0; }}
  .model-chip {{ padding: 6px 16px; border-radius: 20px; color: white;
                  font-size: .85em; font-weight: 600; }}
  footer {{ text-align: center; padding: 30px; color: #999; font-size: .85em; }}
</style>
</head>
<body>
<header>
  <h1>🔍 Benchmark OCR — Cartes grises espagnoles</h1>
  <p>Généré le {generated} · Modèles : {len(models)} · Champs : {len(ALL_VARS)}</p>
</header>
<div class="container">
  <div class="models">
    {''.join(f'<span class="model-chip" style="background:{model_colors[m]}">{m}</span>' for m in models)}
  </div>
  <div class="grid">
    <div class="card">
      <h2>⚡ Latence moyenne / doc</h2>
      <table>{rows_lat}</table>
    </div>
    <div class="card">
      <h2>📋 Taux de remplissage (champs extraits)</h2>
      <table>{rows_fill}</table>
    </div>
    <div class="card">
      <h2>🤖 Score LLM-as-a-judge</h2>
      <table>{rows_llm}</table>
    </div>
    <div class="card">
      <h2>🎯 Accuracy avec Ground Truth</h2>
      <table>{rows_gt}</table>
    </div>
  </div>
</div>
<footer>Benchmark OCR — Cartes grises espagnoles</footer>
</body>
</html>"""


def run():
    print("📊 Génération du rapport final...\n")

    data = {
        "models":     ACTIVE_MODELS,
        "latencies":  load_latencies(),
        "fill_rates": load_fill_rates(),
        "llm_scores": load_llm_scores(),
        "gt_scores":  load_gt_scores(),
        "generated":  datetime.now().strftime("%d/%m/%Y %H:%M"),
    }

    # HTML
    html = generate_html_report(data)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    html_path = RESULTS_REPORT / f"report_{ts}.html"
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"🌐 Rapport HTML → {html_path}")

    # CSV récapitulatif
    csv_path = RESULTS_REPORT / f"summary_{ts}.csv"
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "model", "avg_latency_s", "fill_rate_pct",
            "llm_judge_score_pct", "gt_accuracy_pct"
        ])
        writer.writeheader()
        for model in ACTIVE_MODELS:
            writer.writerow({
                "model":               model,
                "avg_latency_s":       round(data["latencies"].get(model, 0), 3),
                "fill_rate_pct":       round(data["fill_rates"].get(model, 0), 1),
                "llm_judge_score_pct": round(data["llm_scores"].get(model, 0), 1),
                "gt_accuracy_pct":     round(data["gt_scores"].get(model, 0), 1),
            })
    print(f"📄 Résumé CSV    → {csv_path}")


if __name__ == "__main__":
    run()

In [ ]:
https://github.com/curiousily/AI-Bootcamp/blob/master/GLM-OCR.ipynb

In [ ]:
"""
Rapport final — Comparaison parseurs × modèles OCR.

Lit tous les dossiers results_benchmark_*/ au même niveau,
extrait les métriques de chaque combinaison parseur+modèle
et produit deux CSV :

  1. report_global.csv     — une ligne par combinaison parseur+modèle
  2. report_by_field.csv   — une ligne par combinaison parseur+modèle+champ

Structure attendue :
    results_benchmark_regex/
        raw/
            paddleocr/
                carte_001.json
                carte_002.json
                ...
            markitdown/
                ...
    results_benchmark_mistral/
        raw/
            paddleocr/
                ...

Usage :
    python phase4_report.py
    python phase4_report.py --root /chemin/vers/dossier/parent
    python phase4_report.py --root . --prefix results_benchmark
"""

import argparse
import csv
import json
from collections import defaultdict
from pathlib import Path

# ─── CONFIG ───────────────────────────────────────────────────────────────────
# Adapte si besoin
DEFAULT_ROOT   = Path(".")          # dossier parent qui contient les results_benchmark_*/
BENCHMARK_PREFIX = "results_benchmark_"
OUTPUT_DIR     = Path("report_final")

# Champs à évaluer — remplace par ton ALL_VARS réel
try:
    from config import ALL_VARS, GROUND_TRUTH_DIR
except ImportError:
    ALL_VARS          = []   # ⚠️ sera détecté dynamiquement depuis les JSON
    GROUND_TRUTH_DIR  = Path("ground_truth")


# ─── CHARGEMENT ───────────────────────────────────────────────────────────────

def detect_fields_from_json(json_path: Path) -> list[str]:
    """Détecte dynamiquement les champs depuis un JSON si ALL_VARS est vide."""
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)
    return list(data.get("fields", {}).keys())


def load_all_benchmarks(root: Path, prefix: str) -> dict:
    """
    Parcourt tous les dossiers results_benchmark_*/ et charge les JSON raw.

    Retourne :
    {
      "regex": {
        "paddleocr": {
          "carte_001": {"fields": {...}, "latency_s": 1.2, "error": None},
          ...
        },
        "markitdown": {...}
      },
      "mistral": {...}
    }
    """
    benchmarks = {}
    dirs = sorted([d for d in root.iterdir()
                   if d.is_dir() and d.name.startswith(prefix)])

    if not dirs:
        raise FileNotFoundError(
            f"Aucun dossier '{prefix}*' trouvé dans {root.resolve()}"
        )

    for bench_dir in dirs:
        parser_name = bench_dir.name.replace(prefix, "")  # ex: "regex" ou "mistral"
        raw_dir = bench_dir / "raw"
        if not raw_dir.exists():
            print(f"  ⚠️  Pas de dossier raw/ dans {bench_dir.name} — ignoré")
            continue

        benchmarks[parser_name] = {}
        for model_dir in sorted(raw_dir.iterdir()):
            if not model_dir.is_dir():
                continue
            model_name = model_dir.name
            benchmarks[parser_name][model_name] = {}

            for json_file in sorted(model_dir.glob("*.json")):
                doc_id = json_file.stem
                with open(json_file, encoding="utf-8") as f:
                    data = json.load(f)
                benchmarks[parser_name][model_name][doc_id] = {
                    "fields":    data.get("fields", {}),
                    "latency_s": data.get("latency_s", 0),
                    "error":     data.get("error"),
                }

        print(f"  ✅ {bench_dir.name}  →  "
              f"{len(benchmarks[parser_name])} modèles  |  "
              f"{sum(len(v) for v in benchmarks[parser_name].values())} docs total")

    return benchmarks


def load_ground_truth(gt_dir: Path) -> dict:
    """Retourne {doc_id: {field: value}}."""
    gt = {}
    if not gt_dir.exists():
        return gt
    for p in gt_dir.glob("*.json"):
        with open(p, encoding="utf-8") as f:
            gt[p.stem] = json.load(f)
    return gt


# ─── MÉTRIQUES ────────────────────────────────────────────────────────────────

def cer(ref: str, hyp: str) -> float:
    ref = (ref or "").upper().strip()
    hyp = (hyp or "").upper().strip()
    if not ref:
        return 0.0 if not hyp else 1.0
    m, n = len(ref), len(hyp)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, n + 1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            dp[j] = min(dp[j]+1, dp[j-1]+1, prev[j-1]+cost)
    return round(dp[n] / m, 4)


def cross_agreement(docs: dict, all_parsers_models: list[tuple], doc_id: str, field: str) -> float:
    """
    Pour un doc+champ, calcule le % de combinaisons parseur+modèle
    qui sont d'accord avec la valeur majoritaire.
    """
    values = []
    for parser, model, model_docs in all_parsers_models:
        val = model_docs.get(doc_id, {}).get("fields", {}).get(field, "")
        if val:
            values.append(val.upper().strip())
    if not values:
        return 0.0
    majority = max(set(values), key=values.count)
    return sum(1 for v in values if v == majority) / len(values)


# ─── ANALYSE ──────────────────────────────────────────────────────────────────

def analyze(benchmarks: dict, gt: dict, fields: list[str]) -> tuple[list, list]:
    """
    Retourne (global_rows, field_rows) pour les deux CSV.
    """
    # Liste plate de toutes les combinaisons parseur+modèle
    all_combos = [
        (parser, model, model_docs)
        for parser, models in benchmarks.items()
        for model, model_docs in models.items()
    ]

    global_rows = []
    field_rows  = []

    for parser, model, model_docs in all_combos:
        combo_name = f"{parser}+{model}"
        print(f"  Analyse : {combo_name}")

        # Accumulateurs
        latencies        = []
        errors           = 0
        fill_per_field   = defaultdict(list)   # field → [0/1]
        gt_exact         = defaultdict(int)
        gt_cer_vals      = defaultdict(list)
        gt_n             = defaultdict(int)
        cross_agr_vals   = []

        all_doc_ids = list(model_docs.keys())

        for doc_id, doc_data in model_docs.items():
            if doc_data.get("error"):
                errors += 1
                continue

            lat = doc_data.get("latency_s", 0)
            if lat:
                latencies.append(float(lat))

            extracted = doc_data.get("fields", {})

            # Remplissage par champ
            for field in fields:
                val = extracted.get(field, "")
                fill_per_field[field].append(1 if val and str(val).strip() else 0)

            # Ground truth
            if doc_id in gt:
                for field, ref in gt[doc_id].items():
                    if ref is None or field not in fields:
                        continue
                    hyp = extracted.get(field, "")
                    gt_exact[field]    += int((ref or "").upper().strip() ==
                                              (hyp or "").upper().strip())
                    gt_cer_vals[field].append(cer(ref, hyp))
                    gt_n[field]        += 1

            # Cross-validation (accord avec majorité toutes combos)
            field_agreements = []
            for field in fields:
                agr = cross_agreement(model_docs, all_combos, doc_id, field)
                field_agreements.append(agr)
            if field_agreements:
                cross_agr_vals.append(
                    sum(field_agreements) / len(field_agreements)
                )

        # ── Calcul métriques globales ──────────────────────────────────────
        n_docs      = len(all_doc_ids)
        avg_lat     = sum(latencies) / len(latencies) if latencies else 0
        error_rate  = errors / n_docs * 100 if n_docs else 0

        all_fills   = [v for vals in fill_per_field.values() for v in vals]
        fill_rate   = sum(all_fills) / len(all_fills) * 100 if all_fills else 0

        cross_agr   = (sum(cross_agr_vals) / len(cross_agr_vals) * 100
                       if cross_agr_vals else None)

        gt_total_n  = sum(gt_n.values())
        gt_total_ex = sum(gt_exact.values())
        all_cers    = [c for cers in gt_cer_vals.values() for c in cers]
        gt_acc      = gt_total_ex / gt_total_n * 100 if gt_total_n else None
        gt_cer_avg  = sum(all_cers) / len(all_cers) if all_cers else None

        # ── Score global composite ────────────────────────────────────────
        speed_score = max(0, 100 - avg_lat * 10)

        if gt_acc is None and cross_agr is None:
            score = fill_rate * 0.80 + speed_score * 0.20
        elif gt_acc is None:
            score = fill_rate * 0.45 + (cross_agr or 0) * 0.45 + speed_score * 0.10
        elif cross_agr is None:
            score = fill_rate * 0.45 + (gt_acc or 0) * 0.45 + speed_score * 0.10
        else:
            score = (fill_rate * 0.25 + (cross_agr or 0) * 0.25 +
                     (gt_acc or 0) * 0.40 + speed_score * 0.10)

        global_rows.append({
            "combo":                  combo_name,
            "parser":                 parser,
            "ocr_model":              model,
            "n_docs":                 n_docs,
            "n_errors":               errors,
            "error_rate_pct":         round(error_rate, 1),
            "avg_latency_s":          round(avg_lat, 3),
            "fill_rate_pct":          round(fill_rate, 1),
            "cross_agreement_pct":    round(cross_agr, 1) if cross_agr is not None else "N/A",
            "gt_accuracy_pct":        round(gt_acc, 1)    if gt_acc    is not None else "N/A",
            "gt_avg_cer":             round(gt_cer_avg, 4) if gt_cer_avg is not None else "N/A",
            "gt_n_fields_evaluated":  gt_total_n,
            "SCORE_GLOBAL":           round(score, 1),
        })

        # ── Métriques par champ ───────────────────────────────────────────
        for field in fields:
            fills   = fill_per_field.get(field, [])
            fill_f  = sum(fills) / len(fills) * 100 if fills else 0
            gtn     = gt_n[field]
            gt_acc_f  = gt_exact[field] / gtn * 100 if gtn else None
            cers_f    = gt_cer_vals[field]
            cer_f     = sum(cers_f) / len(cers_f) if cers_f else None
            field_rows.append({
                "combo":           combo_name,
                "parser":          parser,
                "ocr_model":       model,
                "field":           field,
                "fill_rate_pct":   round(fill_f, 1),
                "gt_accuracy_pct": round(gt_acc_f, 1) if gt_acc_f is not None else "N/A",
                "gt_avg_cer":      round(cer_f, 4)    if cer_f    is not None else "N/A",
                "gt_n":            gtn,
            })

    # Tri par score décroissant
    global_rows.sort(key=lambda x: float(x["SCORE_GLOBAL"]), reverse=True)
    return global_rows, field_rows


# ─── EXPORT CSV ───────────────────────────────────────────────────────────────

def write_csvs(global_rows: list, field_rows: list, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)

    global_path = output_dir / "report_global.csv"
    with open(global_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(global_rows[0].keys()))
        writer.writeheader()
        writer.writerows(global_rows)

    field_path = output_dir / "report_by_field.csv"
    with open(field_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(field_rows[0].keys()))
        writer.writeheader()
        writer.writerows(field_rows)

    return global_path, field_path


# ─── DASHBOARD HTML ───────────────────────────────────────────────────────────

# Palette : une couleur par parseur (jusqu'à 8 parseurs)
_PARSER_PALETTE = [
    {"bar": "#378ADD", "border": "#185FA5"},
    {"bar": "#1D9E75", "border": "#0F6E56"},
    {"bar": "#D85A30", "border": "#993C1D"},
    {"bar": "#D4537E", "border": "#993556"},
    {"bar": "#BA7517", "border": "#854F0B"},
    {"bar": "#639922", "border": "#3B6D11"},
    {"bar": "#7F77DD", "border": "#534AB7"},
    {"bar": "#888780", "border": "#5F5E5A"},
]

_METRICS_DEF = [
    ("avg_latency_s",       "Latence moyenne",      "s",    True),
    ("fill_rate_pct",       "Taux de remplissage",  "%",    False),
    ("cross_agreement_pct", "Cross-validation",     "%",    False),
    ("gt_accuracy_pct",     "GT accuracy",          "%",    False),
    ("gt_avg_cer",          "CER moyen",            "",     True),
    ("SCORE_GLOBAL",        "Score global",         "/100", False),
]


def generate_html_dashboard(global_rows: list, output_dir: Path) -> Path:
    """
    Génère un fichier HTML interactif avec graphes groupés parseur × modèle.
    Un onglet par métrique, barres côte à côte par parseur pour chaque modèle.
    """
    # Déduit parseurs et modèles depuis les données réelles
    parsers = list(dict.fromkeys(r["parser"]    for r in global_rows))
    models  = list(dict.fromkeys(r["ocr_model"] for r in global_rows))

    # Index des données : data[parser][model] = row
    data_index = defaultdict(dict)
    for row in global_rows:
        data_index[row["parser"]][row["ocr_model"]] = row

    # Couleurs par parseur
    parser_colors = {p: _PARSER_PALETTE[i % len(_PARSER_PALETTE)]
                     for i, p in enumerate(parsers)}

    # Synthèse meilleur parseur / modèle
    parser_scores = defaultdict(list)
    model_scores  = defaultdict(list)
    for row in global_rows:
        parser_scores[row["parser"]].append(float(row["SCORE_GLOBAL"]))
        model_scores[row["ocr_model"]].append(float(row["SCORE_GLOBAL"]))
    parser_avg = {p: sum(v)/len(v) for p, v in parser_scores.items()}
    model_avg  = {m: sum(v)/len(v) for m, v in model_scores.items()}
    best_combo  = global_rows[0]
    best_parser = max(parser_avg, key=parser_avg.get)
    best_model  = max(model_avg,  key=model_avg.get)

    # ── Sérialise les données pour JS ─────────────────────────────────────────
    def safe(val):
        try:
            return float(val)
        except (ValueError, TypeError):
            return 0.0

    js_data = {}
    for p in parsers:
        js_data[p] = {}
        for m in models:
            row = data_index[p].get(m, {})
            js_data[p][m] = {k: safe(row.get(k, 0)) for k, *_ in _METRICS_DEF}

    js_parsers       = json.dumps(parsers)
    js_models        = json.dumps(models)
    js_data_str      = json.dumps(js_data)
    js_metrics       = json.dumps([
        {"key": k, "label": lbl, "unit": u, "lowerBetter": lb}
        for k, lbl, u, lb in _METRICS_DEF
    ])
    js_colors        = json.dumps({p: parser_colors[p] for p in parsers})
    js_best_combo    = json.dumps(best_combo["combo"])
    js_best_parser   = json.dumps(best_parser)
    js_best_model    = json.dumps(best_model)
    js_parser_avg    = json.dumps({p: round(v, 1) for p, v in parser_avg.items()})
    js_model_avg     = json.dumps({m: round(v, 1) for m, v in model_avg.items()})
    js_best_score    = json.dumps(float(best_combo["SCORE_GLOBAL"]))
    js_best_p_score  = json.dumps(round(parser_avg[best_parser], 1))
    js_best_m_score  = json.dumps(round(model_avg[best_model], 1))

    html = f"""<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Benchmark OCR — Cartes grises ES</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<style>
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:system-ui,sans-serif;background:#f5f7fa;color:#333;font-size:14px}}
header{{background:#1a2744;color:#fff;padding:20px 32px}}
header h1{{font-size:1.3em;font-weight:500;margin-bottom:4px}}
header p{{opacity:.65;font-size:.85em}}
.container{{max-width:1100px;margin:24px auto;padding:0 20px}}
.winner-row{{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:12px;margin-bottom:24px}}
.winner-card{{background:#fff;border:0.5px solid #e0e0e0;border-radius:10px;padding:14px 16px}}
.winner-tag{{font-size:11px;color:#185FA5;background:#E6F1FB;padding:2px 8px;border-radius:6px;display:inline-block;margin-bottom:8px}}
.winner-name{{font-size:15px;font-weight:500}}
.winner-score{{font-size:12px;color:#888;margin-top:2px}}
.tabs{{display:flex;gap:4px;flex-wrap:wrap;margin-bottom:20px}}
.tab{{font-size:12px;padding:5px 14px;border-radius:6px;border:0.5px solid #ccc;background:transparent;color:#666;cursor:pointer}}
.tab.active{{background:#1a2744;color:#fff;border-color:#1a2744;font-weight:500}}
.legend{{display:flex;gap:14px;flex-wrap:wrap;margin-bottom:12px}}
.legend-item{{display:flex;align-items:center;gap:5px;font-size:12px;color:#666}}
.legend-dot{{width:10px;height:10px;border-radius:2px}}
.chart-wrap{{position:relative;width:100%;background:#fff;border:0.5px solid #e0e0e0;border-radius:10px;padding:16px}}
.synthesis{{display:grid;grid-template-columns:1fr 1fr;gap:16px;margin-top:24px}}
.synth-card{{background:#fff;border:0.5px solid #e0e0e0;border-radius:10px;padding:14px 16px}}
.synth-card h3{{font-size:13px;font-weight:500;color:#555;margin-bottom:10px}}
.synth-row{{display:flex;justify-content:space-between;align-items:center;padding:4px 0;border-bottom:0.5px solid #f0f0f0;font-size:13px}}
.synth-row:last-child{{border-bottom:none}}
.synth-val{{font-weight:500;color:#1a2744}}
.note{{font-size:11px;color:#999;margin-top:16px;line-height:1.6}}
</style>
</head>
<body>
<header>
  <h1>Benchmark OCR — Cartes grises espagnoles</h1>
  <p>Comparaison parseurs × modèles OCR</p>
</header>
<div class="container">

  <div class="winner-row" id="winnerRow"></div>
  <div class="tabs" id="tabBar"></div>
  <div class="legend" id="legend"></div>
  <div class="chart-wrap"><div id="chartWrap" style="position:relative"></div></div>

  <div class="synthesis">
    <div class="synth-card">
      <h3>Score moyen par parseur</h3>
      <div id="parserSynth"></div>
    </div>
    <div class="synth-card">
      <h3>Score moyen par modèle OCR</h3>
      <div id="modelSynth"></div>
    </div>
  </div>

  <p class="note">
    Score global = taux de remplissage (25%) + accord cross-validation (25%) + GT accuracy (40%) + rapidité (10%).
    Les métriques N/A sont exclues et les poids redistribués. Latence et CER : plus bas = mieux.
  </p>
</div>

<script>
const PARSERS    = {js_parsers};
const MODELS     = {js_models};
const DATA       = {js_data_str};
const METRICS    = {js_metrics};
const COLORS     = {js_colors};
const BEST_COMBO  = {js_best_combo};
const BEST_PARSER = {js_best_parser};
const BEST_MODEL  = {js_best_model};
const PARSER_AVG  = {js_parser_avg};
const MODEL_AVG   = {js_model_avg};
const BEST_SCORE  = {js_best_score};
const BEST_P_SCORE = {js_best_p_score};
const BEST_M_SCORE = {js_best_m_score};

let activeMetric = METRICS[5].key;
let currentChart = null;

function buildWinners() {{
  document.getElementById('winnerRow').innerHTML = `
    <div class="winner-card"><span class="winner-tag">Meilleure combo</span>
      <div class="winner-name">${{BEST_COMBO}}</div>
      <div class="winner-score">${{BEST_SCORE.toFixed(1)}} / 100</div></div>
    <div class="winner-card"><span class="winner-tag">Meilleur parseur</span>
      <div class="winner-name">${{BEST_PARSER}}</div>
      <div class="winner-score">moy. ${{BEST_P_SCORE.toFixed(1)}} / 100</div></div>
    <div class="winner-card"><span class="winner-tag">Meilleur modèle OCR</span>
      <div class="winner-name">${{BEST_MODEL}}</div>
      <div class="winner-score">moy. ${{BEST_M_SCORE.toFixed(1)}} / 100</div></div>`;
}}

function buildLegend() {{
  document.getElementById('legend').innerHTML = PARSERS.map(p =>
    `<span class="legend-item">
      <span class="legend-dot" style="background:${{COLORS[p].bar}}"></span>${{p}}
    </span>`).join('');
}}

function buildTabs() {{
  const bar = document.getElementById('tabBar');
  METRICS.forEach(m => {{
    const btn = document.createElement('button');
    btn.className = 'tab' + (m.key === activeMetric ? ' active' : '');
    btn.textContent = m.label;
    btn.onclick = () => {{ activeMetric = m.key; renderChart(); updateTabs(); }};
    bar.appendChild(btn);
  }});
}}

function updateTabs() {{
  document.querySelectorAll('.tab').forEach((t, i) =>
    t.classList.toggle('active', METRICS[i].key === activeMetric));
}}

function renderChart() {{
  const metric   = METRICS.find(m => m.key === activeMetric);
  const wrap     = document.getElementById('chartWrap');
  const h        = Math.max(MODELS.length * 72 + 60, 200);
  wrap.innerHTML = `<canvas id="mainChart" role="img"
    aria-label="Graphe ${{metric.label}} par modèle et parseur"
    style="display:block"></canvas>`;
  wrap.style.height = h + 'px';

  if (currentChart) {{ currentChart.destroy(); currentChart = null; }}

  const datasets = PARSERS.map(p => ({{
    label: p,
    data:  MODELS.map(m => DATA[p]?.[m]?.[activeMetric] ?? 0),
    backgroundColor: COLORS[p].bar + 'CC',
    borderColor:     COLORS[p].border,
    borderWidth: 1,
    borderRadius: 3,
  }}));

  currentChart = new Chart(document.getElementById('mainChart'), {{
    type: 'bar',
    data: {{ labels: MODELS, datasets }},
    options: {{
      indexAxis: 'y',
      responsive: true,
      maintainAspectRatio: false,
      plugins: {{
        legend: {{ display: false }},
        tooltip: {{
          callbacks: {{
            label: ctx => ` ${{ctx.dataset.label}}: ${{ctx.parsed.x.toFixed(
              metric.key === 'gt_avg_cer' ? 3 : 1)}}${{metric.unit}}`
          }}
        }}
      }},
      scales: {{
        x: {{
          beginAtZero: true,
          grid: {{ color: 'rgba(0,0,0,0.05)' }},
          ticks: {{
            font: {{ size: 11 }}, color: '#888',
            callback: v => v + metric.unit
          }}
        }},
        y: {{
          ticks: {{ font: {{ size: 12 }}, color: '#555' }},
          grid: {{ display: false }}
        }}
      }}
    }}
  }});
}}

function buildSynthesis() {{
  document.getElementById('parserSynth').innerHTML =
    Object.entries(PARSER_AVG).sort((a,b) => b[1]-a[1]).map(([p,v]) =>
      `<div class="synth-row">
        <span style="display:flex;align-items:center;gap:6px">
          <span style="width:8px;height:8px;border-radius:2px;background:${{COLORS[p].bar}};display:inline-block"></span>
          ${{p}}
        </span>
        <span class="synth-val">${{v.toFixed(1)}} / 100</span>
      </div>`).join('');

  document.getElementById('modelSynth').innerHTML =
    Object.entries(MODEL_AVG).sort((a,b) => b[1]-a[1]).map(([m,v]) =>
      `<div class="synth-row"><span>${{m}}</span>
        <span class="synth-val">${{v.toFixed(1)}} / 100</span>
      </div>`).join('');
}}

buildWinners();
buildLegend();
buildTabs();
renderChart();
buildSynthesis();
</script>
</body>
</html>"""

    html_path = output_dir / "dashboard.html"
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html)
    return html_path


# ─── AFFICHAGE TERMINAL ───────────────────────────────────────────────────────

def print_ranking(global_rows: list):
    print("\n" + "═" * 70)
    print("  CLASSEMENT FINAL — PARSEURS × MODÈLES OCR")
    print("═" * 70)

    icons = ["🥇", "🥈", "🥉"] + ["  "] * 20
    for i, row in enumerate(global_rows):
        print(f"\n  {icons[i]}  {row['combo'].upper()}")
        print(f"       Score global      : {row['SCORE_GLOBAL']:>6} / 100")
        print(f"       Latence moyenne   : {row['avg_latency_s']:>6}s")
        print(f"       Taux remplissage  : {row['fill_rate_pct']:>6}%")
        print(f"       Cross-validation  : {str(row['cross_agreement_pct']):>6}%")
        print(f"       GT accuracy       : {str(row['gt_accuracy_pct']):>6}%  "
              f"(CER {row['gt_avg_cer']})")
        print(f"       Erreurs           : {row['n_errors']}/{row['n_docs']} docs")

    parser_scores = defaultdict(list)
    model_scores  = defaultdict(list)
    for row in global_rows:
        parser_scores[row["parser"]].append(float(row["SCORE_GLOBAL"]))
        model_scores[row["ocr_model"]].append(float(row["SCORE_GLOBAL"]))
    parser_avg = {p: sum(v)/len(v) for p, v in parser_scores.items()}
    model_avg  = {m: sum(v)/len(v) for m, v in model_scores.items()}

    print("\n" + "─" * 70)
    print("  SYNTHÈSE")
    print("─" * 70)
    print(f"\n  Meilleur parseur  : {max(parser_avg, key=parser_avg.get).upper()}")
    for p, avg in sorted(parser_avg.items(), key=lambda x: -x[1]):
        print(f"    {p:<20} {avg:.1f} / 100")
    print(f"\n  Meilleur modèle OCR : {max(model_avg, key=model_avg.get).upper()}")
    for m, avg in sorted(model_avg.items(), key=lambda x: -x[1]):
        print(f"    {m:<20} {avg:.1f} / 100")
    print("\n  Score = remplissage 25% + cross-valid. 25% + GT acc. 40% + rapidité 10%\n")


# ─── MAIN ─────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--root",   type=Path, default=DEFAULT_ROOT,
                        help="Dossier parent contenant les results_benchmark_*/")
    parser.add_argument("--prefix", type=str,  default=BENCHMARK_PREFIX,
                        help="Préfixe des dossiers benchmark")
    parser.add_argument("--gt",     type=Path, default=GROUND_TRUTH_DIR,
                        help="Dossier ground truth")
    parser.add_argument("--output", type=Path, default=OUTPUT_DIR,
                        help="Dossier de sortie")
    args = parser.parse_args()

    print(f"\n🔍 Recherche des benchmarks dans : {args.root.resolve()}")
    print(f"   Préfixe : {args.prefix}\n")

    benchmarks = load_all_benchmarks(args.root, args.prefix)
    gt         = load_ground_truth(args.gt)
    print(f"\n   Ground truth : {len(gt)} docs annotés")

    fields = ALL_VARS
    if not fields:
        for models in benchmarks.values():
            for model_docs in models.values():
                for doc_data in model_docs.values():
                    detected = list(doc_data["fields"].keys())
                    if detected:
                        fields = detected
                        break
                if fields:
                    break
            if fields:
                break
        print(f"   Champs détectés automatiquement : {fields}")

    if not fields:
        print("❌ Impossible de détecter les champs. Vérifie tes JSON raw.")
        return

    print("\n📊 Calcul des métriques...\n")
    global_rows, field_rows = analyze(benchmarks, gt, fields)

    # CSV
    global_path, field_path = write_csvs(global_rows, field_rows, args.output)

    # Dashboard HTML
    html_path = generate_html_dashboard(global_rows, args.output)

    # Classement terminal
    print_ranking(global_rows)

    print(f"  💾  report_global.csv   → {global_path}")
    print(f"  💾  report_by_field.csv → {field_path}")
    print(f"  🌐  dashboard.html      → {html_path}\n")


if __name__ == "__main__":
    main()